# U01_P — Sequences as Text — Solutions
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 1:** Bioinformatics and Internet
**Type:** Practice Solutions notebook · **Time:** 50 min · **Difficulty:** ★★☆
**Goes with:** Lectures L1–L2, Practical S1
**Before this:** U01_C1

In [1]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U01_P"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

Python 3.11.18 | Biopython 1.88


### Ex 1.1 ★☆☆ — Calculate length and base counts of human insulin mRNA
**Biology:** Insulin (INS) is a peptide hormone critical for glucose homeostasis. Its mRNA sequence is stored in data/sequences/ins_human_mrna.fasta.  
**Task:** Load `ins_human_mrna.fasta` using `SeqIO.read`, store its sequence length in `ins_len`, and count how many 'A' nucleotides it contains in `ins_a_count`.  
**Expected:** `ins_len` as an integer (around 465 bp), `ins_a_count` as an integer.

In [2]:
from Bio import SeqIO

ins_path = DATA_DIR / "sequences" / "ins_human_mrna.fasta"
ins_record = SeqIO.read(ins_path, "fasta")
ins_len = len(ins_record.seq)
ins_a_count = ins_record.seq.count("A")

In [3]:
# Check — do not edit
assert isinstance(ins_len, int), "ins_len must be an integer"
assert isinstance(ins_a_count, int), "ins_a_count must be an integer"
assert ins_len > 400 and ins_len < 500, f"Expected length ~465 bp, got {ins_len}"
assert ins_a_count > 50, f"ins_a_count too low: {ins_a_count}"
print("Ex 1.1 passed")

Ex 1.1 passed


### Ex 1.2 ★☆☆ — Calculate GC percentage of human insulin mRNA
**Biology:** GC content influences mRNA stability and secondary structure folding energy.  
**Task:** Compute the GC percentage of human insulin mRNA and assign it to `ins_gc_percent` as a float rounded to 2 decimal places.  
**Expected:** `ins_gc_percent` as a float between 50.0 and 70.0.

In [4]:
ins_seq = ins_record.seq
g_count = ins_seq.count("G")
c_count = ins_seq.count("C")
ins_gc_percent = round(((g_count + c_count) / len(ins_seq)) * 100, 2)

In [5]:
# Check — do not edit
assert isinstance(ins_gc_percent, float), "ins_gc_percent should be a float"
assert 55.0 <= ins_gc_percent <= 65.0, f"Expected GC% between 55% and 65%, got {ins_gc_percent}"
print("Ex 1.2 passed")

Ex 1.2 passed


### Ex 1.3 ★☆☆ — Extract the 5' leader of HBB mRNA
**Biology:** The 5' untranslated region precedes the start codon (AUG) and regulates translation initiation.  
**Task:** Load `hbb_human_mrna.fasta`. The first 50 bases represent the 5' UTR. Extract them into variable `hbb_leader`.  
**Expected:** A string or Seq of length 50 starting with 'ACATTTG'.

In [6]:
hbb_record = SeqIO.read(DATA_DIR / "sequences" / "hbb_human_mrna.fasta", "fasta")
hbb_leader = str(hbb_record.seq[:50])

In [7]:
# Check — do not edit
assert len(hbb_leader) == 50, f"Expected length 50, got {len(hbb_leader)}"
assert hbb_leader.startswith("ACATTTG"), "hbb_leader should start with ACATTTG"
print("Ex 1.3 passed")

Ex 1.3 passed


### Ex 1.4 ★★☆ — Compare base composition between HBB coding sequence and mRNA
**Biology:** Untranslated regions (UTRs) frequently exhibit different nucleotide compositions compared to protein-coding exons.  
**Task:** Calculate the GC% of `hbb_human_cds.fasta` (`cds_gc`) and compare it to the full mRNA GC% (`mrna_gc`). Store the difference (`cds_gc - mrna_gc`) in `gc_diff`.  
**Expected:** `gc_diff` as a float rounded to 2 decimal places.

In [8]:
cds_record = SeqIO.read(DATA_DIR / "sequences" / "hbb_human_cds.fasta", "fasta")
cds_gc = round(((cds_record.seq.count("G") + cds_record.seq.count("C")) / len(cds_record.seq)) * 100, 2)
mrna_gc = round(((hbb_record.seq.count("G") + hbb_record.seq.count("C")) / len(hbb_record.seq)) * 100, 2)
gc_diff = round(cds_gc - mrna_gc, 2)

In [9]:
# Check — do not edit
assert isinstance(gc_diff, float), "gc_diff must be a float"
assert abs(gc_diff) < 10.0, "Difference between CDS and mRNA GC% should be modest"
print("Ex 1.4 passed")

Ex 1.4 passed


### Ex 1.5 ★★☆ — Parse a multi-FASTA file and identify sequence lengths
**Biology:** Multiple sequences in a file represent gene families, orthologues, or synthetic benchmarks.  
**Task:** Using `SeqIO.parse`, read `data/sequences/toy_sequences.fasta`. Store the number of records in `num_records`, and store a dictionary mapping record IDs to lengths in `length_dict`.  
**Expected:** `num_records` integer >= 2, `length_dict` mapping str -> int.

In [10]:
toy_path = DATA_DIR / "sequences" / "toy_sequences.fasta"
length_dict = {}
for rec in SeqIO.parse(toy_path, "fasta"):
    length_dict[rec.id] = len(rec.seq)
num_records = len(length_dict)

In [11]:
# Check — do not edit
assert isinstance(num_records, int) and num_records >= 2, "Expected at least 2 toy sequences"
assert all(isinstance(v, int) for v in length_dict.values()), "All lengths must be integers"
print("Ex 1.5 passed")

Ex 1.5 passed


### Ex 1.6 ★★☆ — Confirm start codon complement on the minus strand
**Biology:** If a gene is transcribed on the minus strand, the template DNA contains 3'-TAC-5', whose reverse complement in 5'->3' orientation is ATG.  
**Task:** Take the first 3 nucleotides of `cds_record.seq` (the canonical start codon). Compute its reverse complement in `rc_start`. Verify it equals 'CAT'.  
**Expected:** `rc_start` as a string equal to 'CAT'.

In [12]:
start_codon = cds_record.seq[:3]
rc_start = str(start_codon.reverse_complement())

In [13]:
# Check — do not edit
assert rc_start == "CAT", f"Reverse complement of ATG should be CAT, got {rc_start}"
print("Ex 1.6 passed")

Ex 1.6 passed


### Ex 1.7 ★★★ — Sliding window GC calculation across HBB mRNA
**Biology:** Local GC fluctuations indicate structural features, CpG islands, or codon usage shifts.  
**Task:** Calculate the GC percentage in non-overlapping 100 bp windows along `hbb_record.seq`. Store the window GC percentages as a list of floats in `window_gcs`.  
**Expected:** A list of floats with length equal to `len(hbb_record.seq) // 100`.

In [14]:
window_size = 100
full_dna = str(hbb_record.seq)
window_gcs = []

for i in range(0, len(full_dna) - window_size + 1, window_size):
    chunk = full_dna[i:i + window_size]
    gc_val = round(((chunk.count("G") + chunk.count("C")) / window_size) * 100, 2)
    window_gcs.append(gc_val)

In [15]:
# Check — do not edit
assert isinstance(window_gcs, list) and len(window_gcs) == 6, f"Expected 6 windows of 100 bp, got {len(window_gcs)}"
assert all(30.0 <= x <= 75.0 for x in window_gcs), "Window GC percentages should fall in 30-75% range"
print("Ex 1.7 passed")

Ex 1.7 passed


## 5. Reflection Questions
1. Why does Python's 0-based indexing require biological coordinates to be converted when comparing against GenBank flat files?
2. Why is the reverse complement necessary when analyzing genes encoded on the opposite DNA strand, rather than just the simple string reverse (`seq[::-1]`)?